# Chapter 4: Retrieval-Augmented Generation

*Small Language Models in Practice — Haji Gul*

> Giving a small model fresh knowledge without retraining; embeddings and vector
search explained simply; and a complete RAG pipeline over your own documents
using LanceDB — chunk, embed, retrieve, answer.

---

*Lecture notes mirroring the book. Run the setup cell, then work top-to-bottom. Swap model ids freely.*

## Setup
Uncomment what this chapter needs.

In [ ]:
# %pip install -q transformers datasets accelerate torch
# Chapter-specific installs appear in shell cells below.

## The idea in one paragraph

A fine-tuned model still only knows what was in its training data. **RAG**
fixes this at query time: store your documents as searchable vectors, retrieve
the few chunks most relevant to the question, and paste them into the prompt as
context. The model then answers *from your data* — no retraining, and you
can update the knowledge base any time.

> **Embeddings, plainly.** An **embedding** is a list of numbers representing a piece of text so that
similar meanings land close together in space. To find relevant chunks we embed
the question and return the stored chunks whose vectors are nearest. That nearest
search is what a **vector database** does fast.

## Step 1: chunk your documents

Long documents must be split into retrievable pieces. Overlap keeps sentences
from being cut awkwardly across chunks.

In [ ]:
def chunk_text(text, size=500, overlap=50):
    chunks, start = [], 0
    while start < len(text):
        end = start + size
        chunks.append(text[start:end])
        start = end - overlap
    return chunks

docs = [
    "Small language models run locally and keep data private. ...",
    "LoRA trains tiny adapter matrices instead of full weights. ...",
    # In practice: read .txt/.md/.pdf files and concatenate their text.
]

chunks = [c for d in docs for c in chunk_text(d)]
print(f"{len(chunks)} chunks ready")

## Step 2: embed the chunks

We use a small, fast sentence-embedding model that runs on CPU.

In [ ]:
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
vectors = embedder.encode(chunks, normalize_embeddings=True)
print(vectors.shape)   # (n_chunks, 384)

## Step 3: store them in a vector database

LanceDB is file-based — no server to run. We store each chunk with its vector.

In [ ]:
import lancedb

db = lancedb.connect("./rag_db")
data = [
    {"text": chunk, "vector": vec.tolist()}
    for chunk, vec in zip(chunks, vectors)
]
table = db.create_table("docs", data=data, mode="overwrite")

## Step 4: retrieve relevant context

Embed the question with the *same* model, then ask the table for the nearest
chunks.

In [ ]:
def retrieve(question, k=3):
    q_vec = embedder.encode(question, normalize_embeddings=True)
    hits = table.search(q_vec.tolist()).limit(k).to_list()
    return [h["text"] for h in hits]

context = retrieve("How does LoRA save memory?")
for c in context:
    print("-", c[:80])

## Step 5: generate the grounded answer

Stitch retrieved context into the prompt and let the model answer from it.

In [ ]:
from transformers import pipeline

gen = pipeline("text-generation",
               model="Qwen/Qwen2.5-0.5B-Instruct", device_map="auto")

def rag_answer(question):
    context = "\n\n".join(retrieve(question))
    messages = [
        {"role": "system",
         "content": "Answer using ONLY the context. If unsure, say so."},
        {"role": "user",
         "content": f"Context:\n{context}\n\nQuestion: {question}"},
    ]
    out = gen(messages, max_new_tokens=150, do_sample=False)
    return out[0]["generated_text"][-1]["content"]

print(rag_answer("How does LoRA save memory?"))

> **Tip.** Two levers dominate RAG quality: **chunk size** (too big dilutes relevance,
too small loses context) and **k** (how many chunks you retrieve). Tune
these before reaching for a bigger model — they matter more.

## Recap and exercise

You built a full RAG pipeline: chunk, embed, store, retrieve, and generate a
grounded answer — all locally, with no retraining.

**Exercise.** Point the loader at a folder of your own `.txt` files,
rebuild the table, and ask three domain questions. Add one question the documents
do *not* answer and confirm the model admits it does not know.